# 3D U-Net Inference Pipeline for Cell Tracking
This notebook uses a trained 3D U-Net to predict Gaussian heatmaps and track cells.
**Usage:** Add the output dataset from the `train-3d-unet` notebook to this Kaggle notebook. Update `MODEL_WEIGHTS_PATH` to point to `model_best.pth`.



In [ ]:
!pip install -q "monai[ignite, torchvision]" blosc2


In [ ]:
import os
import json
import time
import blosc2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
import monai
from monai.networks.nets import UNet
from monai.inferers import sliding_window_inference

print('All imports successful.')



In [ ]:
# Configuration
TEST_DIR = '/kaggle/input/competitions/biohub-cell-tracking-during-development/test'
MODEL_WEIGHTS_PATH = '/kaggle/input/your-dataset-name/model_best.pth'  # UPDATE THIS

# Physical voxel scale (um per voxel)
SCALE = np.array([1.625, 0.40625, 0.40625])  # Z, Y, X

# Inference parameters
PATCH_SIZE = (32, 128, 128)
HEATMAP_THRESH = 0.3  # Minimum heatmap intensity to be considered a cell
NMS_SIZE_XY = 4
NMS_SIZE_Z = 1

# Linking parameters
MAX_LINK_DISTANCE = 12.0
GAP_LINK_DISTANCE = 15.0
GAP_FRAMES = 1
DIVISION_DISTANCE = 18.0
MIN_TRACK_LEN_DIVISION = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print('Configuration loaded.')



In [ ]:
# Load Model
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm="batch",
).to(device)

if os.path.exists(MODEL_WEIGHTS_PATH):
    model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=device))
    print("Loaded custom trained weights.")
else:
    print("WARNING: MODEL_WEIGHTS_PATH not found! Using random initialization for demonstration.")

model.eval()



In [ ]:
# Detection Functions
def detect_peaks_heatmap(heatmap, nms_size_z, nms_size_xy, threshold):
    footprint_size = (2 * nms_size_z + 1, 2 * nms_size_xy + 1, 2 * nms_size_xy + 1)
    local_max = maximum_filter(heatmap, size=footprint_size)
    peaks_mask = (heatmap == local_max) & (heatmap > threshold)
    peak_coords = np.argwhere(peaks_mask)
    return peak_coords.astype(np.float64)

def detect_cells_unet(vol):
    """Predicts heatmap using sliding window and extracts cell coordinates."""
    vol_min, vol_max = vol.min(), vol.max()
    if vol_max > vol_min:
        vol = (vol - vol_min) / (vol_max - vol_min)
        
    # Prepare tensor
    vol_tensor = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Sliding window inference
        heatmap_tensor = sliding_window_inference(
            inputs=vol_tensor,
            roi_size=PATCH_SIZE,
            sw_batch_size=4,
            predictor=model,
            overlap=0.25
        )
        
    heatmap = heatmap_tensor.squeeze().cpu().numpy()
    
    # Extract peaks
    centroids = detect_peaks_heatmap(heatmap, NMS_SIZE_Z, NMS_SIZE_XY, HEATMAP_THRESH)
    return centroids



In [ ]:
# Linking Functions (KD-Tree Hungarian)
def link_frames_hungarian(prev_phys, curr_phys, prev_ids, curr_ids, max_dist):
    if len(prev_phys) == 0 or len(curr_phys) == 0:
        return [], set(), set()
    n_prev = len(prev_phys)
    n_curr = len(curr_phys)
    if n_prev * n_curr < 4_000_000:
        dist = np.linalg.norm(prev_phys[:, None, :] - curr_phys[None, :, :], axis=2)
        dist[dist > max_dist] = 1e6
        row_ind, col_ind = linear_sum_assignment(dist)
        edges = []
        matched_prev = set()
        matched_curr = set()
        for ri, ci in zip(row_ind, col_ind):
            if dist[ri, ci] <= max_dist:
                edges.append((prev_ids[ri], curr_ids[ci]))
                matched_prev.add(prev_ids[ri])
                matched_curr.add(curr_ids[ci])
        return edges, matched_prev, matched_curr
    else:
        tree = cKDTree(curr_phys)
        neighbors = tree.query_ball_point(prev_phys, r=max_dist)
        row_list, col_list, dist_list = [], [], []
        for i, nbrs in enumerate(neighbors):
            for j in nbrs:
                d = np.linalg.norm(prev_phys[i] - curr_phys[j])
                row_list.append(i)
                col_list.append(j)
                dist_list.append(d)
        if not row_list:
            return [], set(), set()
        unique_rows = sorted(set(row_list))
        unique_cols = sorted(set(col_list))
        row_map = {r: i for i, r in enumerate(unique_rows)}
        col_map = {c: i for i, c in enumerate(unique_cols)}
        cost = np.full((len(unique_rows), len(unique_cols)), 1e6)
        for r, c, d in zip(row_list, col_list, dist_list):
            cost[row_map[r], col_map[c]] = d
        ri, ci = linear_sum_assignment(cost)
        edges = []
        matched_prev = set()
        matched_curr = set()
        for r, c in zip(ri, ci):
            if cost[r, c] <= max_dist:
                orig_r = unique_rows[r]
                orig_c = unique_cols[c]
                edges.append((prev_ids[orig_r], curr_ids[orig_c]))
                matched_prev.add(prev_ids[orig_r])
                matched_curr.add(curr_ids[orig_c])
        return edges, matched_prev, matched_curr

def detect_divisions(parent_phys, parent_ids, child_phys, child_ids, matched_parent_ids, matched_child_ids, max_dist, track_lengths):
    division_edges = []
    unmatched_parent_mask = np.array([pid not in matched_parent_ids for pid in parent_ids])
    unmatched_child_mask = np.array([cid not in matched_child_ids for cid in child_ids])
    if not np.any(unmatched_parent_mask) or np.sum(unmatched_child_mask) < 2:
        return division_edges
    unmatched_parent_phys = parent_phys[unmatched_parent_mask]
    unmatched_parent_ids = [pid for pid, m in zip(parent_ids, unmatched_parent_mask) if m]
    unmatched_child_phys = child_phys[unmatched_child_mask]
    unmatched_child_ids = [cid for cid, m in zip(child_ids, unmatched_child_mask) if m]
    if len(unmatched_child_phys) < 2:
        return division_edges
    child_tree = cKDTree(unmatched_child_phys)
    used_children = set()
    for i, pid in enumerate(unmatched_parent_ids):
        if track_lengths.get(pid, 0) < MIN_TRACK_LEN_DIVISION:
            continue
        nearby = child_tree.query_ball_point(unmatched_parent_phys[i], r=max_dist)
        nearby = [n for n in nearby if unmatched_child_ids[n] not in used_children]
        if len(nearby) == 2:
            c1_phys = unmatched_child_phys[nearby[0]]
            c2_phys = unmatched_child_phys[nearby[1]]
            sister_dist = np.linalg.norm(c1_phys - c2_phys)
            if sister_dist < max_dist * 1.5:
                cid1 = unmatched_child_ids[nearby[0]]
                cid2 = unmatched_child_ids[nearby[1]]
                division_edges.append((pid, cid1))
                division_edges.append((pid, cid2))
                used_children.add(cid1)
                used_children.add(cid2)
    return division_edges

def gap_close(lost_tracks, curr_phys, curr_ids, matched_curr_ids, max_dist, scale):
    gap_edges = []
    reconnected_ids = set()
    unmatched_mask = np.array([cid not in matched_curr_ids for cid in curr_ids])
    if not np.any(unmatched_mask) or not lost_tracks:
        return gap_edges, reconnected_ids
    unmatched_curr_phys = curr_phys[unmatched_mask]
    unmatched_curr_ids = [cid for cid, m in zip(curr_ids, unmatched_mask) if m]
    if len(unmatched_curr_phys) == 0:
        return gap_edges, reconnected_ids
    lost_ids = list(lost_tracks.keys())
    lost_phys = np.array([lost_tracks[lid][0] for lid in lost_ids])
    if len(lost_phys) == 0:
        return gap_edges, reconnected_ids
    edges, matched_lost, _ = link_frames_hungarian(lost_phys, unmatched_curr_phys, lost_ids, unmatched_curr_ids, max_dist)
    return edges, matched_lost



In [ ]:
# Main Processing Loop
test_folder_names = sorted(d.replace('.zarr', '') for d in os.listdir(TEST_DIR) if d.endswith('.zarr'))
print(f'Found {len(test_folder_names)} test samples.')

all_rows = []
total_start = time.time()

for sample_idx, folder_name in enumerate(test_folder_names):
    sample_start = time.time()
    zarr_path = os.path.join(TEST_DIR, folder_name + '.zarr')

    with open(os.path.join(zarr_path, '0', 'zarr.json')) as f:
        arr_meta = json.load(f)
    shape = tuple(arr_meta['shape'])
    dtype = np.dtype(arr_meta['data_type'])
    n_t = shape[0]
    vol_shape = shape[1:]
    print(f'\n[{sample_idx+1}/{len(test_folder_names)}] {folder_name}')

    node_id_counter = 1
    frame_phys = {}
    track_lengths = {}
    lost_tracks = {}
    sample_edges = []
    sample_nodes = []

    for t in range(n_t):
        chunk_path = os.path.join(zarr_path, '0', 'c', str(t), '0', '0', '0')
        with open(chunk_path, 'rb') as fh:
            compressed = fh.read()
        decompressed = blosc2.decompress(compressed)
        vol = np.frombuffer(decompressed, dtype=dtype).reshape(vol_shape).astype(np.float32)

        # UNet Detection
        centroids_voxel = detect_cells_unet(vol)

        curr_ids = []
        curr_phys_dict = {}
        for cent in centroids_voxel:
            nid = node_id_counter
            node_id_counter += 1
            z_int = max(0, min(vol_shape[0] - 1, int(round(cent[0]))))
            y_int = max(0, min(vol_shape[1] - 1, int(round(cent[1]))))
            x_int = max(0, min(vol_shape[2] - 1, int(round(cent[2]))))
            curr_ids.append(nid)
            curr_phys_dict[nid] = cent * SCALE
            track_lengths[nid] = 1
            sample_nodes.append({
                'dataset': folder_name,
                'row_type': 'node',
                'node_id': nid,
                't': t,
                'z': z_int,
                'y': y_int,
                'x': x_int,
                'source_id': -1,
                'target_id': -1,
            })

        frame_phys[t] = curr_phys_dict

        # Linking
        if t > 0 and (t - 1) in frame_phys:
            prev_phys_dict = frame_phys[t - 1]
            prev_ids = list(prev_phys_dict.keys())
            if prev_ids and curr_ids:
                prev_phys_arr = np.array([prev_phys_dict[pid] for pid in prev_ids])
                curr_phys_arr = np.array([curr_phys_dict[cid] for cid in curr_ids])

                edges, matched_prev, matched_curr = link_frames_hungarian(
                    prev_phys_arr, curr_phys_arr, prev_ids, curr_ids, MAX_LINK_DISTANCE
                )
                for src, tgt in edges:
                    track_lengths[tgt] = track_lengths.get(src, 1) + 1
                    sample_edges.append({
                        'dataset': folder_name,
                        'row_type': 'edge',
                        'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                        'source_id': src, 'target_id': tgt,
                    })

                if lost_tracks:
                    gap_edges, reconnected = gap_close(
                        lost_tracks, curr_phys_arr, curr_ids, matched_curr, GAP_LINK_DISTANCE, SCALE
                    )
                    for src, tgt in gap_edges:
                        track_lengths[tgt] = track_lengths.get(src, 1) + 1
                        sample_edges.append({
                            'dataset': folder_name,
                            'row_type': 'edge',
                            'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                            'source_id': src, 'target_id': tgt,
                        })
                        matched_curr.add(tgt)
                    for rid in reconnected:
                        lost_tracks.pop(rid, None)

                div_edges = detect_divisions(
                    prev_phys_arr, prev_ids, curr_phys_arr, curr_ids,
                    matched_prev, matched_curr, DIVISION_DISTANCE, track_lengths
                )
                for src, tgt in div_edges:
                    track_lengths[tgt] = 1
                    sample_edges.append({
                        'dataset': folder_name,
                        'row_type': 'edge',
                        'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                        'source_id': src, 'target_id': tgt,
                    })

                new_lost = {}
                for pid in prev_ids:
                    if pid not in matched_prev and not any(s == pid for s, _ in div_edges):
                        new_lost[pid] = (prev_phys_dict[pid], 1)
                updated_lost = {}
                for lid, (coords, age) in lost_tracks.items():
                    if lid not in reconnected and age < GAP_FRAMES:
                        updated_lost[lid] = (coords, age + 1)
                updated_lost.update(new_lost)
                lost_tracks = updated_lost
            else:
                lost_tracks = {}

        if t >= 2 and (t - 2) in frame_phys:
            del frame_phys[t - 2]

        if (t + 1) % 10 == 0 or t == n_t - 1:
            print(f'  Frame {t+1}/{n_t}: {len(centroids_voxel)} cells, {len(sample_edges)} edges so far')

    all_rows.extend(sample_nodes)
    all_rows.extend(sample_edges)
    
    elapsed = time.time() - sample_start
    print(f'  DONE: {len(sample_nodes)} nodes, {len(sample_edges)} edges ({elapsed:.1f}s)')



In [ ]:
# Create Submission
submission = pd.DataFrame(all_rows)
submission = submission[['dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']]
submission.index = range(len(submission))
submission.index.name = 'id'

int_cols = ['node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
for col in int_cols:
    submission[col] = submission[col].astype(int)

submission.to_csv('submission.csv')
print(f'Submission written: {len(submission)} rows')

